In [1]:
import gdown
import pandas as pd
url1 = 'https://drive.google.com/uc?id=1fYGb-n114IGWZDcBKE0jzc3BWSLCcsVQ'
gdown.download(url1, 'data1.csv', quiet=False)
url2 = 'https://drive.google.com/uc?id=1eX9an3tSypvrFn_fiPReYHTca5GpJaJG'
gdown.download(url2, 'data2.csv', quiet=False)


df1 = pd.read_csv('data1.csv')
df2 = pd.read_csv('data2.csv')



Downloading...
From (original): https://drive.google.com/uc?id=1fYGb-n114IGWZDcBKE0jzc3BWSLCcsVQ
From (redirected): https://drive.google.com/uc?id=1fYGb-n114IGWZDcBKE0jzc3BWSLCcsVQ&confirm=t&uuid=411891ef-c59b-4d3c-9691-6d563d3c2854
To: /content/data1.csv
100%|██████████| 794M/794M [00:11<00:00, 68.7MB/s]
Downloading...
From: https://drive.google.com/uc?id=1eX9an3tSypvrFn_fiPReYHTca5GpJaJG
To: /content/data2.csv
100%|██████████| 6.16k/6.16k [00:00<00:00, 12.9MB/s]
/tmp/ipykernel_2141/1460247288.py:9: DtypeWarning: Columns (11,27,29,53) have mixed types. Specify dtype option on import or set low_memory=False.
  df1 = pd.read_csv('data1.csv')


In [ ]:
import warnings
warnings.filterwarnings("ignore")
import re
import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split, KFold
from sklearn.impute import SimpleImputer
from sklearn.ensemble import RandomForestRegressor, GradientBoostingRegressor
from sklearn.metrics import r2_score, mean_absolute_error, mean_squared_error
print("=" * 70)
print("1. Combining datasets")
print("=" * 70)

df_raw = pd.concat([df1, df2], ignore_index=True)

print("Initial shape:", df_raw.shape)


df = df_raw.copy()


def fa_to_en(value):
    """
    Convert Persian/Arabic digits to English digits.
    """
    if pd.isna(value):
        return value

    value = str(value)

    translation = str.maketrans(
        "۰۱۲۳۴۵۶۷۸۹٠١٢٣٤٥٦٧٨٩",
        "01234567890123456789"
    )

    return value.translate(translation)


def clean_numeric(series):
    """
    Convert mixed numeric/string columns to numeric.
    """
    return pd.to_numeric(
        series.astype(str)
        .map(fa_to_en)
        .str.replace(",", "", regex=False)
        .str.replace("٬", "", regex=False)
        .str.replace("٫", ".", regex=False)
        .str.replace(r"[^\d\.\-]", "", regex=True),
        errors="coerce"
    )


def safe_text(series):
    """
    Convert text column to lowercase string.
    """
    return series.fillna("").astype(str).str.lower()


df.columns = (
    df.columns
    .astype(str)
    .str.strip()
)


for col in ["title", "description", "cat2_slug", "credit_mode", "price_mode"]:
    if col in df.columns:
        df[col] = df[col].fillna("").astype(str)

numeric_candidates = [
    "price_value",
    "transformable_price",
    "transformable_credit",
    "transformed_credit",
    "transformable_rent",
    "transformed_rent",
    "credit_value",
    "rent_value",
    "building_size",
    "land_size",
    "construction_year",
    "floor",
    "total_floors",
    "rooms",
    "latitude",
    "longitude",
    "radius",
    "capacity",
    "price_per_meter"
]

for col in numeric_candidates:
    if col in df.columns:
        df[col] = clean_numeric(df[col])
print("\n" + "=" * 70)
print("2. Removing unwanted listings")
print("=" * 70)

before = len(df)

unwanted_keywords = [
    "خوابگاه",
    "روزانه",
    "روز",
    "زائر",
    "زیبایی",
    "صندلی",
    "اشتراکی",
    "همخانه",
    "اتاق",
    "dormitory",
    "daily",
    "pilgrim",
    "beauty",
    "shared room",
    "roommate"
]

if "title" in df.columns:

    title_text = safe_text(df["title"])

    unwanted_mask = pd.Series(False, index=df.index)

    for keyword in unwanted_keywords:
        unwanted_mask |= title_text.str.contains(
            keyword,
            regex=False,
            na=False
        )

    df = df.loc[~unwanted_mask].copy()
if "credit_mode" in df.columns:

    credit_text = safe_text(df["credit_mode"])

    invalid_credit = (
        credit_text.str.contains("رایگان", regex=False, na=False)
        |
        credit_text.str.contains("free", regex=False, na=False)
    )

    df = df.loc[~invalid_credit].copy()
print("Creating target price...")

target = pd.Series(np.nan, index=df.index, dtype=float)
if "price_value" in df.columns:
    target = df["price_value"].copy()
if "transformable_price" in df.columns:

    mask = target.isna() | (target <= 0)

    target.loc[mask] = df.loc[
        mask,
        "transformable_price"
    ]
if "credit_value" in df.columns or "rent_value" in df.columns:

    credit = (
        df["credit_value"]
        if "credit_value" in df.columns
        else pd.Series(0, index=df.index)
    )

    rent = (
        df["rent_value"]
        if "rent_value" in df.columns
        else pd.Series(0, index=df.index)
    )

    credit = credit.fillna(0)
    rent = rent.fillna(0)

    equivalent_credit = (
        credit
        +
        (rent / 3_000_000) * 100_000_000
    )

    calculated_price = equivalent_credit * 10

    mask = (
        (target.isna() | (target <= 0))
        &
        (calculated_price > 0)
    )

    target.loc[mask] = calculated_price.loc[mask]


df["target_price"] = target
df = df[
    df["target_price"].notna()
    &
    (df["target_price"] > 0)
].copy()
MIN_TOTAL_PRICE = 100_000_000

df = df[
    df["target_price"] >= MIN_TOTAL_PRICE
].copy()
if "building_size" in df.columns:

    df = df[
        df["building_size"].notna()
        &
        (df["building_size"] >= 20)
        &
        (df["building_size"] <= 700)
    ].copy()
df["price_per_meter"] = (
    df["target_price"]
    /
    df["building_size"]
)


df = df[
    df["price_per_meter"].notna()
    &
    np.isfinite(df["price_per_meter"])
    &
    (df["price_per_meter"] > 0)
].copy()
MIN_PRICE_PER_METER = 1_000_000
df = df[
    df["price_per_meter"] >= MIN_PRICE_PER_METER
].copy()
print("After basic cleaning:", df.shape)
print("\nRemoving extreme outliers...")

target_low = df["target_price"].quantile(0.005)
target_high = df["target_price"].quantile(0.995)

ppm_low = df["price_per_meter"].quantile(0.005)
ppm_high = df["price_per_meter"].quantile(0.995)

df = df[
    df["target_price"].between(
        target_low,
        target_high
    )
    &
    df["price_per_meter"].between(
        ppm_low,
        ppm_high
    )
].copy()


print("After outlier removal:", df.shape)
if "construction_year" in df.columns:

    year = df["construction_year"].copy()
    year = pd.to_numeric(year, errors="coerce")
    mask_gregorian = year > 1700

    year.loc[mask_gregorian] = (
        year.loc[mask_gregorian] - 621
    )

    df["building_age"] = (
        1403 - year
    )

    df["building_age"] = df["building_age"].clip(
        lower=0,
        upper=80
    )

else:

    df["building_age"] = np.nan

print("\nCreating numerical features...")


if "building_size" in df.columns:

    df["log_building_size"] = np.log1p(
        df["building_size"].clip(lower=0)
    )

    df["building_size_sq"] = (
        df["building_size"] ** 2
    )


if "land_size" in df.columns and "building_size" in df.columns:

    df["land_to_building"] = (
        df["land_size"]
        /
        df["building_size"].replace(0, np.nan)
    )

    df["building_minus_land"] = (
        df["building_size"]
        -
        df["land_size"]
    )


if "rooms" in df.columns and "building_size" in df.columns:

    df["size_per_room"] = (
        df["building_size"]
        /
        df["rooms"].replace(0, np.nan)
    )

    df["rooms_per_100m"] = (
        df["rooms"]
        /
        df["building_size"].replace(0, np.nan)
        * 100
    )


if "floor" in df.columns:

    df["floor_abs"] = df["floor"].abs()

    df["is_ground_floor"] = (
        df["floor"] == 0
    ).astype(int)


if "floor" in df.columns and "total_floors" in df.columns:

    df["is_top_floor"] = (
        df["floor"]
        ==
        df["total_floors"]
    ).astype(int)

    df["floor_ratio"] = (
        df["floor"]
        /
        df["total_floors"].replace(0, np.nan)
    )


if "latitude" in df.columns:

    df["latitude_sq"] = (
        df["latitude"] ** 2
    )


if "longitude" in df.columns:

    df["longitude_sq"] = (
        df["longitude"] ** 2
    )


if "latitude" in df.columns and "longitude" in df.columns:

    df["lat_lon_interaction"] = (
        df["latitude"]
        *
        df["longitude"]
    )

amenity_keywords = [
    "parking",
    "elevator",
    "storage",
    "balcony",
    "انباری",
    "پارکینگ",
    "آسانسور",
    "بالکن"
]

amenity_cols = []

for col in df.columns:

    col_lower = str(col).lower()

    if any(
        keyword in col_lower
        for keyword in amenity_keywords
    ):
        amenity_cols.append(col)


amenity_cols = list(dict.fromkeys(amenity_cols))

if len(amenity_cols) > 0:

    amenity_numeric = pd.DataFrame(index=df.index)

    for col in amenity_cols:

        values = df[col]

        if values.dtype == object:

            text = values.fillna("").astype(str).str.lower()

            amenity_numeric[col] = (
                text.isin(
                    [
                        "1",
                        "true",
                        "yes",
                        "بله",
                        "دارد",
                        "موجود"
                    ]
                )
            ).astype(int)

        else:

            amenity_numeric[col] = pd.to_numeric(
                values,
                errors="coerce"
            ).fillna(0).gt(0).astype(int)

    df["amenities_count"] = amenity_numeric.sum(axis=1)

else:

    df["amenities_count"] = 0

if "title" in df.columns:

    title = safe_text(df["title"])

    df["title_length"] = title.str.len()

    df["title_word_count"] = (
        title.str.split().str.len()
    )

    title_keywords = {
        "has_parking": [
            "پارکینگ",
            "parking"
        ],
        "has_elevator": [
            "آسانسور",
            "elevator"
        ],
        "has_storage": [
            "انباری",
            "storage"
        ],
        "has_balcony": [
            "بالکن",
            "تراس",
            "balcony",
            "terrace"
        ],
        "has_renovated": [
            "بازسازی",
            "نوسازی",
            "renovated"
        ],
        "has_new": [
            "نوساز",
            "نوساخته",
            "new"
        ],
        "has_furnished": [
            "مبله",
            "furnished"
        ],
        "has_luxury": [
            "لوکس",
            "luxury"
        ],
        "has_sunlight": [
            "نورگیر",
            "sunlight"
        ],
        "has_view": [
            "ویو",
            "نما",
            "view"
        ]
    }

    for feature_name, keywords in title_keywords.items():

        pattern = "|".join(
            re.escape(keyword)
            for keyword in keywords
        )

        df[feature_name] = (
            title.str.contains(
                pattern,
                regex=True,
                na=False
            )
        ).astype(int)

    df["title_digit_count"] = (
        title.str.count(r"\d")
    )

else:

    df["title_length"] = 0
    df["title_word_count"] = 0
    df["title_digit_count"] = 0




if "description" in df.columns:

    description = safe_text(df["description"])

    df["description_length"] = (
        description.str.len()
    )

    df["description_word_count"] = (
        description.str.split().str.len()
    )

else:

    df["description_length"] = 0
    df["description_word_count"] = 0



print("Creating frequency encoding...")

categorical_frequency_cols = [
    "city_slug",
    "neighborhood_slug",
    "cat2_slug",
    "property_type",
    "deed_type",
    "user_type",
    "price_mode",
    "credit_mode"
]

frequency_maps = {}

for col in categorical_frequency_cols:

    if col in df.columns:

        counts = df[col].fillna("__MISSING__").value_counts()

        frequency_maps[col] = counts

        df[col + "_frequency"] = (
            df[col]
            .fillna("__MISSING__")
            .map(counts)
            .fillna(0)
        )
print("\n" + "=" * 70)
print("3. Splitting data")
print("=" * 70)

train_val, test = train_test_split(
    df,
    test_size=0.20,
    random_state=42
)

train, validation = train_test_split(
    train_val,
    test_size=0.20,
    random_state=42
)


print("Train:", train.shape)
print("Validation:", validation.shape)
print("Test:", test.shape)
print("\nCreating leakage-safe target encoding...")


def create_target_encoding(
    train_df,
    validation_df,
    test_df,
    column,
    target_column="price_per_meter",
    n_splits=5,
    smoothing=100
):

    global_mean = train_df[target_column].mean()

    train_encoded = pd.Series(
        global_mean,
        index=train_df.index,
        dtype=float
    )

    kfold = KFold(
        n_splits=n_splits,
        shuffle=True,
        random_state=42
    )

    for fold_train_idx, fold_valid_idx in kfold.split(train_df):

        fold_train = train_df.iloc[fold_train_idx]
        fold_valid = train_df.iloc[fold_valid_idx]

        stats = (
            fold_train
            .groupby(column)[target_column]
            .agg(["mean", "count"])
        )

        smoothed = (
            (
                stats["count"] * stats["mean"]
                +
                smoothing * global_mean
            )
            /
            (
                stats["count"]
                +
                smoothing
            )
        )

        encoded_values = (
            fold_valid[column]
            .map(smoothed)
            .fillna(global_mean)
        )

        train_encoded.iloc[fold_valid_idx] = (
            encoded_values.values
        )

    full_stats = (
        train_df
        .groupby(column)[target_column]
        .agg(["mean", "count"])
    )

    full_smoothed = (
        (
            full_stats["count"] * full_stats["mean"]
            +
            smoothing * global_mean
        )
        /
        (
            full_stats["count"]
            +
            smoothing
        )
    )

    validation_encoded = (
        validation_df[column]
        .map(full_smoothed)
        .fillna(global_mean)
    )

    test_encoded = (
        test_df[column]
        .map(full_smoothed)
        .fillna(global_mean)
    )

    return (
        train_encoded,
        validation_encoded,
        test_encoded
    )


target_encoding_cols = [
    "neighborhood_slug",
    "city_slug",
    "property_type"
]


for col in target_encoding_cols:

    if col not in train.columns:
        continue

    train_enc, val_enc, test_enc = create_target_encoding(
        train,
        validation,
        test,
        col,
        target_column="price_per_meter",
        n_splits=5,
        smoothing=100
    )

    train = train.copy()
    validation = validation.copy()
    test = test.copy()

    train[col + "_target_encoding"] = train_enc
    validation[col + "_target_encoding"] = val_enc
    test[col + "_target_encoding"] = test_enc
if (
    "neighborhood_slug_target_encoding" in train.columns
    and
    "building_size" in train.columns
):

    train["base_price_per_sqm"] = (
        train["neighborhood_slug_target_encoding"]
    )

    validation["base_price_per_sqm"] = (
        validation["neighborhood_slug_target_encoding"]
    )

    test["base_price_per_sqm"] = (
        test["neighborhood_slug_target_encoding"]
    )

    train["estimated_base_price"] = (
        train["base_price_per_sqm"]
        *
        train["building_size"]
    )

    validation["estimated_base_price"] = (
        validation["base_price_per_sqm"]
        *
        validation["building_size"]
    )

    test["estimated_base_price"] = (
        test["base_price_per_sqm"]
        *
        test["building_size"]
    )

    train["log_estimated_base_price"] = np.log1p(
        train["estimated_base_price"].clip(lower=0)
    )

    validation["log_estimated_base_price"] = np.log1p(
        validation["estimated_base_price"].clip(lower=0)
    )

    test["log_estimated_base_price"] = np.log1p(
        test["estimated_base_price"].clip(lower=0)
    )
print("\n" + "=" * 70)
print("4. Preparing model data")
print("=" * 70)


drop_columns = [
    "target_price",
    "price_per_meter",
    "price_value",
    "transformable_price",
    "transformable_credit",
    "transformed_credit",
    "transformable_rent",
    "transformed_rent",
    "credit_value",
    "rent_value",
    "title",
    "description",
    "city_slug",
    "neighborhood_slug",
    "cat2_slug",
    "property_type",
    "deed_type",
    "user_type",
    "price_mode",
    "credit_mode",
    "construction_year",
    "rent_price_on_holidays",
    "rent_price_on_special_days"
]


def prepare_features(data):

    X = data.copy()

    existing_drop_columns = [
        col
        for col in drop_columns
        if col in X.columns
    ]

    X = X.drop(
        columns=existing_drop_columns,
        errors="ignore"
    )

    X = X.select_dtypes(
        include=[np.number]
    )
    X = X.replace(
        [np.inf, -np.inf],
        np.nan
    )

    return X
X_train = prepare_features(train)
X_validation = prepare_features(validation)
X_test = prepare_features(test)
common_columns = (
    X_train.columns
    .intersection(X_validation.columns)
    .intersection(X_test.columns)
)

X_train = X_train[common_columns].copy()
X_validation = X_validation[common_columns].copy()
X_test = X_test[common_columns].copy()
nunique = X_train.nunique()

constant_columns = nunique[
    nunique <= 1
].index.tolist()

if len(constant_columns) > 0:

    X_train = X_train.drop(
        columns=constant_columns
    )

    X_validation = X_validation.drop(
        columns=constant_columns
    )

    X_test = X_test.drop(
        columns=constant_columns
    )


print("Number of features:", X_train.shape[1])


imputer = SimpleImputer(
    strategy="median"
)

X_train_imp = imputer.fit_transform(
    X_train
)

X_validation_imp = imputer.transform(
    X_validation
)

X_test_imp = imputer.transform(
    X_test
)

y_train = train["target_price"].values
y_validation = validation["target_price"].values
y_test = test["target_price"].values
y_train_log = np.log1p(
    y_train
)
print("\n" + "=" * 70)
print("5. Training Random Forest")
print("=" * 70)

rf_model = RandomForestRegressor(
    n_estimators=140,
    max_depth=24,
    min_samples_leaf=5,
    max_features=0.70,
    n_jobs=-1,
    random_state=42
)

rf_model.fit(
    X_train_imp,
    y_train_log
)

rf_val_log = rf_model.predict(
    X_validation_imp
)

rf_val_pred = np.expm1(
    rf_val_log
)

rf_val_pred = np.maximum(
    rf_val_pred,
    0
)


rf_r2 = r2_score(
    y_validation,
    rf_val_pred
)

rf_mae = mean_absolute_error(
    y_validation,
    rf_val_pred
)

rf_mse = mean_squared_error(
    y_validation,
    rf_val_pred
)

rf_rmse = np.sqrt(
    rf_mse
)


print("\nRandom Forest - Validation")
print("R2   :", round(rf_r2, 4))
print("MAE  :", round(rf_mae, 2))
print("MSE  :", round(rf_mse, 2))
print("RMSE :", round(rf_rmse, 2))
print("\n" + "=" * 70)
print("6. Training Gradient Boosting")
print("=" * 70)

gb_model = GradientBoostingRegressor(
    n_estimators=240,
    learning_rate=0.045,
    max_depth=4,
    min_samples_leaf=12,
    subsample=0.85,
    loss="huber",
    random_state=42
)

gb_model.fit(
    X_train_imp,
    y_train_log
)

gb_val_log = gb_model.predict(
    X_validation_imp
)

gb_val_pred = np.expm1(
    gb_val_log
)

gb_val_pred = np.maximum(
    gb_val_pred,
    0
)


gb_r2 = r2_score(
    y_validation,
    gb_val_pred
)

gb_mae = mean_absolute_error(
    y_validation,
    gb_val_pred
)

gb_mse = mean_squared_error(
    y_validation,
    gb_val_pred
)

gb_rmse = np.sqrt(
    gb_mse
)


print("\nGradient Boosting - Validation")
print("R2   :", round(gb_r2, 4))
print("MAE  :", round(gb_mae, 2))
print("MSE  :", round(gb_mse, 2))
print("RMSE :", round(gb_rmse, 2))


print("\n" + "=" * 70)
print("7. Model Selection")
print("=" * 70)

if gb_r2 >= rf_r2:

    best_model_name = "Gradient Boosting"
    best_model = gb_model

    print("Selected model: Gradient Boosting")

else:

    best_model_name = "Random Forest"
    best_model = rf_model

    print("Selected model: Random Forest")
print("\n" + "=" * 70)
print("8. FINAL TEST EVALUATION")
print("=" * 70)

best_test_log = best_model.predict(
    X_test_imp
)

best_test_pred = np.expm1(
    best_test_log
)

best_test_pred = np.maximum(
    best_test_pred,
    0
)


test_r2 = r2_score(
    y_test,
    best_test_pred
)

test_mae = mean_absolute_error(
    y_test,
    best_test_pred
)

test_mse = mean_squared_error(
    y_test,
    best_test_pred
)

test_rmse = np.sqrt(
    test_mse
)


print("\nFINAL RESULT")
print("-" * 50)
print("Model:", best_model_name)
print("R2   :", round(test_r2, 4))
print("MAE  :", round(test_mae, 2))
print("MSE  :", round(test_mse, 2))
print("RMSE :", round(test_rmse, 2))
print("\n" + "=" * 70)
print("9. Top Feature Importances")
print("=" * 70)


if hasattr(best_model, "feature_importances_"):

    importance_df = pd.DataFrame({
        "feature": X_train.columns,
        "importance": best_model.feature_importances_
    })

    importance_df = importance_df.sort_values(
        "importance",
        ascending=False
    )

    print(
        importance_df.head(25).to_string(
            index=False
        )
    )

print("\n" + "=" * 70)
print("PROJECT SUMMARY")
print("=" * 70)

print("Initial dataset size :", len(df_raw))
print("Cleaned dataset size :", len(df))
print("Train size           :", len(train))
print("Validation size      :", len(validation))
print("Test size            :", len(test))
print("Number of features   :", X_train.shape[1])
print("Best model           :", best_model_name)
print("Validation R2        :",
      round(
          max(rf_r2, gb_r2),
          4
      ))
print("Test R2              :",
      round(test_r2, 4))
print("Test MAE             :",
      round(test_mae, 2))
print("Test MSE             :",
      round(test_mse, 2))
print("Test RMSE            :",
      round(test_rmse, 2))

print("\nDone.")



1. Combining datasets
Initial shape: (1000240, 63)

2. Removing unwanted listings
Creating target price...
After basic cleaning: (826915, 65)

Removing extreme outliers...
After outlier removal: (814028, 65)

Creating numerical features...
Creating frequency encoding...

3. Splitting data
Train: (520977, 93)
Validation: (130245, 93)
Test: (162806, 93)

Creating leakage-safe target encoding...

4. Preparing model data
Number of features: 44

5. Training Random Forest

Random Forest - Validation
R2   : 0.6984
MAE  : 2026375197.03
MSE  : 3.5357902542987137e+19
RMSE : 5946251133.53

6. Training Gradient Boosting

Gradient Boosting - Validation
R2   : 0.6541
MAE  : 2302916252.15
MSE  : 4.055403441267759e+19
RMSE : 6368204960.01

7. Model Selection
Selected model: Random Forest

8. FINAL TEST EVALUATION

FINAL RESULT
--------------------------------------------------
Model: Random Forest
R2   : 0.711
MAE  : 1984320795.74
MSE  : 3.2174529765510365e+19
RMSE : 5672259670.14

9. Top Feature Impo